# Grey Literature to Structured Data
Transforming archaeological site records—which often feature a complex mix of typewritten text, handwritten notes, tables (e.g., artifact counts, stratigraphic layers), and varied layouts—into structured data is a valid use case for a Vision-Language Model (VLM).

[Paddle](https://www.paddleocr.ai/latest/en/index.html) ([PaddleOCR-VL](https://www.paddleocr.ai/latest/en/version3.x/pipeline_usage/PaddleOCR-VL.html)) uses a two-stage process: Layout Analysis (identifying tables, paragraphs, and reading order) followed by VLM Recognition (converting those cropped regions into structured formats like Markdown or JSON).

🛑 STOP! Before you run this notebook:
You cannot edit this file directly. Go to the top menu and click File > Save a copy in Drive to create your own working version!

# Setup
Google Colab’s T4 instances currently run CUDA 12.x, but we will explicitly install a stable GPU version of PaddlePaddle alongside the doc-parser toolkit. This will take a few moments. Be patient.

In [ ]:
## Block 1
# 1. Verify that the T4 GPU is attached and accessible
!nvidia-smi

# 2. Install PaddlePaddle matching modern CUDA 12.x (Highly compatible with Colab)
!python -m pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 3. Install the PaddleOCR document parsing extension and the doc handling
!python -m pip install -U "paddleocr[doc-parser]" python-docx

# 4. Colab Kludge: Force reinstall PyTorch with CUDA 12.1
# This restores the correct NCCL 2.19+ dependencies so PyTorch doesn't crash
# when PaddleOCR-VL's transformers try to boot up.
!python -m pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

🛑 STOP! ONCE THAT BLOCK FINISHES, RESTART THE RUNTIME BEFORE CONTINUING 🛑
Because we just updated core system libraries, Google Colab needs to restart its memory to see the changes.

    Go to the top menu and click Runtime > Restart session

    Once it restarts, do NOT run Block 1 again. Skip straight to Block 2.

**After restarting** In this next block we import the library and instantiate the model. Because archaeological records are often photographed in the field or scanned from old binders, they might be skewed or warped. We will enable specific pre-processing flags to handle this.

In [ ]:
# Block 2
from pathlib import Path
from paddleocr import PaddleOCRVL

# 1. Create a directory to store our structured outputs
output_dir = Path("./archaeology_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print("Initializing PaddleOCR-VL Pipeline... (This will download the models on the first run)")

# 2. Instantiate the pipeline.
# Pedagogical Note on Parameters:
# - use_layout_detection=True: (Default) Crucial! It tells the model to chop the page into logical
#   blocks (tables, paragraphs) before reading them, avoiding "hallucinations" on complex pages.
# - use_doc_orientation_classify=True: Great for scanned field notes that might be upside down.
# - use_doc_unwarping=True: Essential if your images are photos of curved pages in a notebook.
pipeline = PaddleOCRVL(
    use_layout_detection=True,
    use_doc_orientation_classify=True,
    use_doc_unwarping=True
)

print("Pipeline ready!")

# Inference & Structuring Data
Now we feed an image to the model. You can point this to a single image, a directory of images, or a PDF. The real magic happens in the output step: we will export the data into JSON (perfect for injecting into a database) and Markdown (perfect for human readability and preserving table structures).

You can drag-and-drop images into the file tray at left, or run the block below. The block below gets a pdf from the archaeology data service and then splits it up into separate images for individual processing, in a new folder called `pdf_images`. The subsequent inference block looks for images in that folder.

In [ ]:
# Get Some Data
!wget -O 1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf https://archaeologydataservice.ac.uk/archiveDS/archiveDownload?t=arch-5014-1/dissemination/documents/site_records/1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf

import pypdfium2 as pdfium
from pathlib import Path

input_pdf_path = "1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf"
output_image_dir = Path("./pdf_images")
output_image_dir.mkdir(exist_ok=True)

print(f"Splitting PDF: {input_pdf_path} into images...")

pdf = pdfium.PdfDocument(input_pdf_path)

for i, page in enumerate(pdf):
    # Render page to PIL image with a high DPI for better quality
    pil_image = page.render(scale=2).to_pil()
    image_path = output_image_dir / f"page_{i+1:03d}.jpg"
    pil_image.save(image_path)

print(f"Successfully split PDF into {len(pdf)} images in '{output_image_dir}'")


**Nb** When you run the code below, Colab might ask if you want to grant access the notebook access to a 'secret'. The secret is a 'token' attached to a user's account (in this case, Hugging Face). Because PaddleOCR-VL downloads its neural network weights from the Hugging Face Hub, you might get asked to share that secret. If you have a hugginface account with tokens set up, you probably already know what to do here (but if not, talk to me). Knowing that a request to download data is coming from an actual user is one way Hugging Face uses to manage it services.

But, you can remain anonymous. You can just hit the 'cancel' button on the 'share notebook secret?' button and the code will try to get the necessary data anyway anonymously anyway. If the IP isn't currently rate-limited, it will succeed! If it doesn't, let me know.

In [ ]:
# Block 3
# Point this to your uploaded archaeological record image (e.g., a JPEG or PNG)
# For this example, you can replace this string with your actual file path in Colab, like "/content/site_record_01.jpg"
input_data = "pdf_images/page_001.jpg" # Demo fallback

print(f"Processing: {input_data}")

# 1. Run the prediction.
# Because we didn't specify an engine, it runs locally on the T4 GPU via PaddlePaddle.
results = pipeline.predict(input_data)

# 2. Iterate through the results (if you passed a directory, this loops through all images)
for res in results:

    # Print the raw bounding boxes, layout classifications, and confidence scores
    print("\n--- Structured Layout Data ---")
    res.print()

    # EXPORT METHOD A: JSON
    # Best for databases. It creates a dictionary mapping bounding boxes to
    # their layout type (e.g., 'table', 'text', 'title') and text content.
    res.save_to_json(save_path=output_dir)

    # EXPORT METHOD B: Markdown
    # Best for LLMs and humans. If your archaeological record has a table of artifact
    # measurements, PaddleOCR-VL reconstructs it as a formatted Markdown table!
    res.save_to_markdown(save_path=output_dir)

    # EXPORT METHOD C: Word Document
    # Generates a visually reconstructed .docx file, attempting to match the original layout.
    res.save_to_word(save_path=output_dir)

print(f"\nSuccess! Check the '{output_dir}' folder in your Colab files menu to see your structured data.")

Optional: Handling Multi-page

In [6]:
# Example for multi-page archaeological PDFs
input_pdf = "1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf"
output = pipeline.predict(input=input_pdf)
pages_res = list(output)

# This powerful function stitches the document logic back together
final_output = pipeline.restructure_pages(
    pages_res,
    merge_tables=True,       # Combines tables that span across page breaks
    relevel_titles=True,     # Reconstructs the hierarchy of Headings
    concatenate_pages=True   # Outputs a single unified file rather than one per page
)

for res in final_output:
    res.save_to_markdown(save_path=output_dir)
    # what would you add to also save to JSON, or to Word?

When reviewing the JSON output, you will notice an attribute called cls_id and label. PaddleOCR-VL categorizes blocks into types like text, table, image, equation, etc. If you are writing a script to auto-ingest site records into a SQL database, you can parse the JSON output to only target label: 'table' to extract artifact logs, while ignoring the label: 'image' (sketches) or text (general notes) blocks.

# Visualizing the Results

You can right-click and select 'download' on any of the output files (in the file tray at left in the 'archaeology_outputs' folder. Then, examine the files with whatever software you have on your local machine. The block below shows a way of examining the json to then export as xls or csv.

In [ ]:
# ==========================================
# BLOCK 5: EXTRACTING THE ARCHAEOLOGICAL TABLE
# ==========================================
import json
import pandas as pd
from IPython.display import display
from pathlib import Path

# We point to the output directory from Block 3
output_dir = Path("./archaeology_outputs")
json_files = list(output_dir.glob("*.json"))

if not json_files:
    print("⚠️ No JSON files found. Make sure you ran Block 4 successfully!")
else:
    target_json = json_files[0]
    print(f"🔍 Loading JSON data from: {target_json.name}\n")

    with open(target_json, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # ---------------------------------------------------------
    # METHOD 1: Targeted Flattening
    # ---------------------------------------------------------
    print("=== Step 1: Isolating the Data Blocks ===")
    # Using 'record_path', we tell Pandas to ignore the metadata and dive directly
    # into the 'parsing_res_list'. It will turn EACH block (title, image, table) into a row!
    df_blocks = pd.json_normalize(data, record_path=['parsing_res_list'])

    # Let's just look at the labels and the first 50 characters of content
    df_preview = df_blocks[['block_label', 'block_id', 'block_content']].copy()
    df_preview['block_content'] = df_preview['block_content'].str[:50] + "..."
    display(df_preview)
    print("\n")

    # ---------------------------------------------------------
    # METHOD 2: The Magic Trick (HTML to DataFrame)
    # ---------------------------------------------------------
    print("=== Step 2: Reconstructing the Site Record Table ===")

    # 1. Filter our dataframe to ONLY grab rows where the AI found a table
    table_rows = df_blocks[df_blocks['block_label'] == 'table']

    if not table_rows.empty:
        # 2. Extract the raw HTML string generated by the Vision-Language Model
        html_string = table_rows.iloc[0]['block_content']

        # 3. Pandas has a built-in HTML parser!
        # It reads the <table>, <tr>, and <td> tags and converts them into a DataFrame
        extracted_tables = pd.read_html(html_string)

        # pd.read_html always returns a list of tables (in case there were multiple).
        # We just want the first one.
        site_record_df = extracted_tables[0]

        # 4. Display the beautiful, structured spreadsheet!
        display(site_record_df)

        # OPTIONAL: You can easily save this directly to an Excel file now!
        # site_record_df.to_excel(output_dir / "Drawing_Register.xlsx", index=False)
        print("\n✅ Table successfully extracted! You can now analyze this data or export it to Excel/CSV.")
    else:
        print("No tables were identified in this document.")